<a href="https://colab.research.google.com/github/Rubix-av/Engage2Value/blob/main/23f2003651_notebook_t22025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

engage_2_value_from_clicks_to_conversions_path = kagglehub.competition_download('engage-2-value-from-clicks-to-conversions')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from scipy.stats import chi2_contingency
import seaborn as sns
import warnings

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message=".*use_inf_as_na.*")
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option('display.max_columns', None)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 🔃 Loading train and test data

### Loading train data to `train`

In [ ]:
train = pd.read_csv("/kaggle/input/engage-2-value-from-clicks-to-conversions/train_data.csv")
train_df_raw = train.copy()

### Loading test data to `test`

In [ ]:
test = pd.read_csv("/kaggle/input/engage-2-value-from-clicks-to-conversions/test_data.csv")
test_df_raw = test.copy()

# 📊 Exploratory Data Analysis (EDA)

# Structure of data

### 1. Train Data has `116023` rows and `52` columns
### 2. Train Data has `15` numerical columns and `37` categorical columns

In [ ]:
train.shape

In [ ]:
train.info()

In [ ]:
print("Total Categorical Column:", 37)
print("Total Numerical Column:", 15)

# Null Values

### 1. `(not set)` in the columns replaced with `NaN`
### 2. Columns with more than `95%` null values

#### `['device.screenResolution', 'trafficSource.adContent', 'trafficSource.keyword', 'trafficSource.adwordsClickInfo.slot', 'device.mobileDeviceBranding', 'device.mobileInputSelector', 'trafficSource.campaign', 'device.mobileDeviceMarketingName', 'device.operatingSystemVersion', 'device.flashVersion', 'geoNetwork.networkLocation', 'trafficSource.adwordsClickInfo.isVideoAd', 'browserMajor', 'device.browserSize', 'trafficSource.adwordsClickInfo.adNetworkType', 'trafficSource.adwordsClickInfo.page', 'geoNetwork.metro', 'device.mobileDeviceModel', 'device.language', 'device.browserVersion', 'device.screenColors'] `

### 3. Columns with null values in range of `50-70%`
#### `['trafficSource.isTrueDirect', 'geoNetwork.region', 'geoNetwork.city', 'trafficSource.referralPath', 'totals.bounces'] `

### 4. Following columns can be imputed like this:
#### - `trafficSource.isTrueDirect -> [nan True] -> [False True]`

#### - `totals.bounces -> [ 1. nan] -> [1. 0]`

#### - `new_visits -> [ 1. nan] -> [1. 0]`

In [ ]:
train.head()

In [ ]:
train.tail()

In [ ]:
# extracting "month, dayofweek, is_weekend, weekofyear, day" out of "date" column
train['date'] = pd.to_datetime(train['date'], format='%Y%m%d')
train['month'] = train['date'].dt.month
train['dayofweek'] = train['date'].dt.dayofweek
train['is_weekend'] = train['dayofweek'].isin([5, 6]).astype(int)
train['weekofyear'] = train['date'].dt.isocalendar().week.astype(int)
train['day'] = train['date'].dt.day
train = train.drop(columns='date', axis=1)

test['date'] = pd.to_datetime(test['date'], format='%Y%m%d')
test['month'] = test['date'].dt.month
test['dayofweek'] = test['date'].dt.dayofweek
test['is_weekend'] = test['dayofweek'].isin([5, 6]).astype(int)
test['weekofyear'] = test['date'].dt.isocalendar().week.astype(int)
test['day'] = test['date'].dt.day
test = test.drop(columns='date', axis=1)

In [ ]:
# replace all occurences of "(not set)" with "NaN"
train = train.replace(["(not set)", "not available in demo dataset", "(not provided)", "unknown"], np.nan)

# replicating for test data
test = test.replace(["(not set)", "not available in demo dataset", "(not provided)", "unknown"], np.nan)

In [ ]:
train.isna().sum()

In [ ]:
data_null_df = pd.DataFrame(train.isna().sum(), columns=["Null Values"])
data_null_df["Null Percentage"] = round(data_null_df["Null Values"]/train.shape[0]*100, 2)

# Columns with null values
data_null_df[data_null_df["Null Percentage"] > 0].sort_values(by="Null Percentage", ascending=False)

In [ ]:
# Creating 3 bins "high_null_val_cols", "medium_null_val_cols" and "low_null_val_cols"

high_null_val_cols = data_null_df[data_null_df["Null Percentage"] > 70].index.tolist()
print("Total columns with high null values:", len(high_null_val_cols))
print(high_null_val_cols, '\n')

medium_null_val_cols = data_null_df[(data_null_df["Null Percentage"] > 50) & (data_null_df["Null Percentage"] < 70)].index.tolist()
print("Total columns with medium null values:", len(medium_null_val_cols))
print(medium_null_val_cols, '\n')

low_null_val_cols = data_null_df[(data_null_df["Null Percentage"] > 0) & (data_null_df["Null Percentage"] < 50)].index.tolist()
print("Total columns with low null values:", len(low_null_val_cols))
print(low_null_val_cols, '\n')

### Dropping the columns with more than `70%` null values

In [ ]:
# Drop all the columns with more than "70%" null values
train = train.drop(columns=high_null_val_cols, axis=1)
test = test.drop(columns=high_null_val_cols, axis=1)

In [ ]:
train.shape

In [ ]:
# replace 'nan' with False and typecast boolean column to int
train["trafficSource.isTrueDirect"] = train["trafficSource.isTrueDirect"].replace(np.nan, False).astype(int)
print(train["trafficSource.isTrueDirect"].unique())

# replace 'nan' with 0
train["totals.bounces"] = train["totals.bounces"].replace(np.nan, 0).astype(int)
print(train["totals.bounces"].unique())

# replace 'nan' with 0
train["new_visits"] = train["new_visits"].replace(np.nan, 0).astype(int)
print(train["new_visits"].unique())

# replicating for test data
test["trafficSource.isTrueDirect"] = test["trafficSource.isTrueDirect"].replace(np.nan, False).astype(int)
test["totals.bounces"] = test["totals.bounces"].replace(np.nan, 0).astype(int)
test["new_visits"] = test["new_visits"].replace(np.nan, 0).astype(int)

In [ ]:
# Retrieving all the columns which have only one unique value
no_variance_cols = [col for col in train.columns if train[col].nunique()==1]
no_variance_cols

In [ ]:
# dropping all the columns with no variance
train = train.drop(columns=no_variance_cols, axis=1)

# replicating on test data
test = test.drop(columns=no_variance_cols, axis=1)

# 🔢 Numerical Columns Analysis

## Observations

### 1. Target variable `purchaseValue` has skewness of `53.91` which is heavily positively skewed
### 2. `userId` has very low variance

In [ ]:
# adding a column "is_purchase" which tells if a purchase has been made or not
train["is_purchase"] = (train["purchaseValue"] > 0).astype(int)

num_cols = train.select_dtypes(exclude=["object"]).columns.tolist()
print(len(num_cols))
print(num_cols, '\n')

cat_cols = train.select_dtypes(include=["object"]).columns.tolist()
print(len(cat_cols))
print(cat_cols)

In [ ]:
# Applying MinMaxScaler on numerical columns for analysis
min_max_scaler = MinMaxScaler()

# Fit and transform the data
num_cols_to_scale = [col for col in num_cols if col not in ["purchaseValue", "is_purchase"]]

# scaling the numerical columns
scaled_values = min_max_scaler.fit_transform(train[num_cols_to_scale])

# creating a dataframe of scaled numerical columns with "purchaseValue"
scaled_nums_df = pd.DataFrame(scaled_values, columns=num_cols_to_scale)
scaled_nums_df["purchaseValue"] = train["purchaseValue"]
scaled_nums_df["is_purchase"] = train["is_purchase"]

In [ ]:
# Printing first 5 rows of "scaled_nums_df"
scaled_nums_df.head()

## Outlier Detection 🔭

### 1. `pageViews` and `totalHits` are positively skewed and there's some potential relation between these two columns
### 2. Though, `sessionId` and `sessionStart` does not have outliers but there seems to be a potential relation between these two
### 3. Target variable `purchaseValue` is heavily right skewed

In [ ]:
sns.set_style("darkgrid")

plt.figure(figsize=(16, 10))
for idx, feature in enumerate(scaled_nums_df.columns, 1):
    plt.subplot(6, 3, idx)
    sns.boxplot(train[feature], orient="h")
    plt.title(feature, fontweight="bold")

plt.tight_layout()
plt.show()

## Distribution Analysis 📊

### 1. Most of the features are moderate to high skewed.
### 2. Some features are nearly symmetric
### 3. Target variable `purchaseValue` is heavily skewed
### 4. `userId` and `sessionId` can be dropped

In [ ]:
sns.set_style("darkgrid")

plt.figure(figsize=(14, len(scaled_nums_df.columns) * 3))
for idx, feature in enumerate(scaled_nums_df.columns, 1):
    plt.subplot(len(scaled_nums_df.columns), 2, idx)
    sns.histplot(train[feature], kde=True)
    plt.title(f"{feature} | Skewness: {round(train[feature].skew(), 2)}")
    print(f"{feature} | Skewness: {round(train[feature].skew(), 2)}")

plt.tight_layout()
plt.show()

## Countplot Analysis 🧮

### 1. No purchase was made for `totals.bounces == 1.0`
### 2. For `trafficSource.isTrueDirect == 0`, most of the users haven't made a purchase
### 3. Most of the new visiting users don't make a purchase, i.e. `new_visits == 1.0`

In [ ]:
plt.figure(figsize=(12, 6))

# Plotting the columns with "is_purchase" as hue
sns.countplot(data=scaled_nums_df, x="trafficSource.isTrueDirect", hue="is_purchase")
plt.title("trafficSource.isTrueDirect Vs is_purchase", fontweight="bold")
plt.xlabel("trafficSource.isTrueDirect", fontweight="bold")
plt.ylabel("Count", fontweight="bold")

# Creating a new dataframe made by grouping the column and target
grouped_data = scaled_nums_df.groupby(["trafficSource.isTrueDirect", "is_purchase"]).size().reset_index(name='count')
grouped_data["count %"] = round(grouped_data["count"]/scaled_nums_df.shape[0]*100, 2)
grouped_data

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(data=scaled_nums_df, x="totals.bounces", hue="is_purchase")
plt.title("totals.bounces Vs is_purchase", fontweight="bold")
plt.xlabel("totals.bounces", fontweight="bold")
plt.ylabel("Count", fontweight="bold")

grouped_data = scaled_nums_df.groupby(["totals.bounces", "is_purchase"]).size().reset_index(name='count')
grouped_data["count %"] = round(grouped_data["count"]/scaled_nums_df.shape[0]*100, 2)
grouped_data

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(data=scaled_nums_df, x="device.isMobile", hue="is_purchase")
plt.title("device.isMobile Vs is_purchase", fontweight="bold")
plt.xlabel("device.isMobile", fontweight="bold")
plt.ylabel("Count", fontweight="bold")

grouped_data = scaled_nums_df.groupby(["device.isMobile", "is_purchase"]).size().reset_index(name='count')
grouped_data["count %"] = round(grouped_data["count"]/scaled_nums_df.shape[0]*100, 2)
grouped_data

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(data=scaled_nums_df, x="new_visits", hue="is_purchase")
plt.title("new_visits Vs is_purchase", fontweight="bold")
plt.xlabel("new_visits", fontweight="bold")
plt.ylabel("Count", fontweight="bold")

grouped_data = scaled_nums_df.groupby(["new_visits", "is_purchase"]).size().reset_index(name='count')
grouped_data["count %"] = round(grouped_data["count"]/scaled_nums_df.shape[0]*100, 2)
grouped_data

## Correlation Matrix 🔎

### 1. `userId` and `sessionId` has very low correlation with other features and target variable so we can drop it.
### 2. `pageViews` and `totalHits` have a high correlation of `0.99` so we can drop one of the columns from each of the pair of columns to prevent multicollinearity.

In [ ]:
plt.figure(figsize=(12,8))

# pearson correlation
sns.heatmap(scaled_nums_df.corr(), annot=True, fmt=".2f", cmap="viridis")

## Correlation Matrix (Spearman Method)

### With `Spearman` correlation method, monotonic relation between the features is captued instead of linear relationship in `Pearson` method. Since, most of the features are skewed and have outliers, `Spearman` correlation will be a better method to identify the important features.

### `trafficSource.isTrueDirect, sessionNumber, pageViews/totalHits, totals.bounces, device.isMobile, new_visits` are some of the important features

In [ ]:
plt.figure(figsize=(12,8))

# spearman correlation
sns.heatmap(scaled_nums_df.corr(method="spearman"), annot=True, fmt=".2f", cmap="viridis")

---

# 🔢 Categorical Columns Analysis

In [ ]:
train[cat_cols].nunique()

### Creating 3 bins for categorical columns based on their `.nunique()` value

In [ ]:
# bin-1 will contain columns from 2-10, bin-2 will contain columns with 11-50 and bin-3 will contain columns with more than 50 .nunique() value
low_cardinality = [col for col in cat_cols if train[col].nunique() <= 10]
print(f"Columns with low cardinality ({len(low_cardinality)}):\n", low_cardinality, "\n")

mid_cardinality = [col for col in cat_cols if (train[col].nunique() > 10) & (train[col].nunique() <= 50)]
print(f"Columns with mid cardinality ({len(mid_cardinality)}):\n", mid_cardinality, "\n")

high_cardinality = [col for col in cat_cols if train[col].nunique() > 50]
print(f"Columns with high cardinality ({len(high_cardinality)}):\n", high_cardinality, "\n")

In [ ]:
plt.figure(figsize=(10,6))
plt.pie([len(low_cardinality), len(mid_cardinality), len(high_cardinality)], labels=["Low", "Mid", "High"], autopct="%1.1f%%")
plt.title("Categorical columns cardinality distribution", fontweight="bold")
plt.show()

## Countplot Analysis

### 1. Categories in `geoCluster & geoNetwork.networkDomain` are equally distributed
### 2. In `trafficSource.medium` column, `affiliate, cpc & cpm` contribtues to less `6%` of the column
### 3. `geoNetwork.continent` column is highly imbalanced having `Americas` contributing to `60.3%` similar imbalanced can be seen with `deviceType` column where `desktop` contributes to `74.5%`

## Low Cardinality Features

In [ ]:
# Creating subplots with 2 rows and 3 columns
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()

total_rows = len(train)

# Iterating through "low_cardinality" where "i" is the index and "col" is the element in list
for i, col in enumerate(low_cardinality):

    # Assign axes to which plot will be plotted
    ax = axes[i]
    plot = sns.countplot(data=train, x=col, ax=ax, width=0.5)

    # Set the title, labels and rotation for "x-axis" labels
    ax.set_title(f"Count plot of {col}", fontweight="bold")
    ax.tick_params(axis='x', labelrotation=45)
    ax.set_ylabel("Count", fontweight="bold")
    ax.set_xlabel(col, fontweight="bold")

    # Add percentages on top of bars
    total = train[col].value_counts().sum()
    for p in plot.patches:
        count = int(p.get_height())
        percentage = count / total * 100
        x = p.get_x() + p.get_width() / 2
        y = p.get_height()
        ax.annotate(f'{percentage:.1f}%', (x, y), ha='center', va='bottom', fontsize=9, fontweight='bold')

# Spacing between subplots rows
fig.subplots_adjust(hspace=0.4)
plt.show()


## Medium Cardinality Features

### 1. `browser` column is highly imbalanced with category `chrome` contributing to `72.8%` values
### 2. Most of the categories in `os` column contributes to less than `1%` values
### 3. `geoNetwork.subContinent` is also highly imbalanced with `North America` having `54.9%` data but rest of the categories are pretty equally distributed

In [ ]:
col = mid_cardinality[0]

plt.figure(figsize=(15, 6))

# plotting the medium cardinality column to the countplot
ax = sns.countplot(data=train, x=col)

# Setting the title, labels and x_tick rotation
ax.set_title(f"Count plot of {col}", fontweight="bold")
ax.tick_params(axis='x', labelrotation=90)
ax.set_ylabel("Count", fontweight="bold")
ax.set_xlabel(col, fontweight="bold")

# Truncate x-tick labels (without modifying DataFrame)
new_labels = []

# looping through the x-tick labels and replace the long string with a shorter one
for label in ax.get_xticklabels():
    text = label.get_text()
    if ";__CT_JOB_ID__" in text:
        fixed = text.replace(text[text.find(";__CT_JOB_ID__"):], ";__CT_JOB_ID__:")
    else:
        fixed = text
    new_labels.append(fixed)

ax.set_xticklabels(new_labels)

# Add percentage annotations
total = train[col].value_counts().sum()
for p in ax.patches:
    count = int(p.get_height())
    percentage = 100 * count / total
    x = p.get_x() + p.get_width() / 2
    y = p.get_height()
    ax.annotate(f'{percentage:.1f}%', (x, y), ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
col = mid_cardinality[1]

plt.figure(figsize=(15, 6))

# plotting the medium cardinality column to the countplot
ax = sns.countplot(data=train, x=col)

# Setting the title, labels and x_tick rotation
ax.set_title(f"Count plot of {col}", fontweight="bold")
ax.tick_params(axis='x', labelrotation=90)
ax.set_ylabel("Count", fontweight="bold")
ax.set_xlabel(col, fontweight="bold")

# Add percentage annotations
total = train[col].value_counts().sum()
for p in ax.patches:
    count = int(p.get_height())
    percentage = 100 * count / total
    x = p.get_x() + p.get_width() / 2
    y = p.get_height()
    ax.annotate(f'{percentage:.1f}%', (x, y), ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
col = mid_cardinality[2]

plt.figure(figsize=(15, 6))


# plotting the medium cardinality column to the countplot
ax = sns.countplot(data=train, x=col)

# Setting the title, labels and x_tick rotation
ax.set_title(f"Count plot of {col}", fontweight="bold")
ax.tick_params(axis='x', labelrotation=90)
ax.set_ylabel("Count", fontweight="bold")
ax.set_xlabel(col, fontweight="bold")

# Add percentage annotations
total = train[col].value_counts().sum()
for p in ax.patches:
    count = int(p.get_height())
    percentage = 100 * count / total
    x = p.get_x() + p.get_width() / 2
    y = p.get_height()
    ax.annotate(f'{percentage:.1f}%', (x, y), ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## High Cardinality Features

In [ ]:
plt.figure(figsize=(10,6))

# Plotting a pie chart for distribution of columns of unique values
plt.pie(train[high_cardinality].nunique().values, labels=train[high_cardinality].nunique().index, autopct="%1.1f%%")
plt.title("Unique Categories in High Cardinality Columns", fontweight="bold")
plt.show()

## High Cardinality Columns Null Vs Non-Null

### * `trafficSource.keyword` has `566` unique categories and `61.9%` null values
### * `trafficSource.referralPath` has `941` unique categories and `63.2%` null values
### Both of these columns have very high cardinality and consists of lot of null values so these columns can be dropped from the dataset.

In [ ]:
# Creating 6 subplots with 2 rows and 3 columns
fig, axes = plt.subplots(2, 3, figsize=(20, 10))

# Loop through the high cardinality columns and plot the pie chart of "Null Vs Non-Null values"
for i, col in enumerate(high_cardinality[:-1]):
    data = [train[col].value_counts().sum(), train[col].isna().sum()]

    axes = axes.flatten()
    axes[i].pie(data, labels=["Non-Null", "Null"], autopct='%1.1f%%')
    axes[i].set_title(f"{col} : {train[col].nunique()}", fontweight="bold")

# Plotting this pie chart in different axis to improve visual clarity
axes[-2].pie([train["trafficSource.referralPath"].value_counts().sum(), train["trafficSource.referralPath"].isna().sum()], labels=["Non-Null", "Null"], autopct='%1.1f%%')
axes[-2].set_title(f"trafficSource.referralPath : {train['trafficSource.referralPath'].nunique()}", fontweight="bold")

# Removing the unused subplot
fig.delaxes(axes[-1])

plt.show()

## Low Cardinality Features Analysis with Target

### 1. Each category in `geoCluster` and `geoNetwork.networkDomain` show nearly identical counts across the two class in `is_purchase (0 and 1)`
### 2. `referral` category in `userChannel` column, has more cases where a purchase was made than the case where purchase wasn't made.
### 3. In `geoNetwork.continent`, most of the purchases are only observed when category is `Americas`, rest of the categories show nearly no instance of purchase.

In [ ]:
# Creating 6 subplots with 3 rows and 2 columns
fig, axes = plt.subplots(3, 2, figsize=(20, 10))
axes = axes.flatten()

# Plotting the countplot of low_cardinality columns
for i, col in enumerate(low_cardinality):
    ax = sns.countplot(data=train, x=col, hue="is_purchase", ax=axes[i], width=0.6)
    ax.set_ylabel("Count", fontweight="bold")
    ax.set_xlabel(col, fontweight="bold")
plt.show()

## Medium Cardinality Features Analysis with Target

### 1. Most of the users use `Chrome` browser and most of the purchases are made by them only.
### 2. Most used `os` is `Windows` but the purchases made are very low. On the other hand, among all the users who use `Macintosh`, more than `50%` of them make a purchase.
### 3. Though `Chrome OS` and `Linux` have less number of users, quite a lot of them make purchases.
### 4. In `geoNetwork.subContinent` column, most of the users belong to `North America` and more than `50%` of them make a purchase. Rest of the users from other subcontinents rarely make a purchase.

In [ ]:
plt.figure(figsize=(20, 8))

# Plotting the countplot of medium_cardinality columns
ax = sns.countplot(data=train, x=mid_cardinality[0], hue="is_purchase")
ax.set_ylabel("Count", fontweight="bold")
ax.set_xlabel(mid_cardinality[0], fontweight="bold")
ax.tick_params(axis='x', labelrotation=90)
ax.set_title(f"{mid_cardinality[0]} countplot", fontweight="bold")

plt.show()

In [ ]:
plt.figure(figsize=(20, 8))

# Plotting the countplot of mid_cardinality columns
ax = sns.countplot(data=train, x=mid_cardinality[1], hue="is_purchase")
ax.set_ylabel("Count", fontweight="bold")
ax.set_xlabel(mid_cardinality[1], fontweight="bold")
ax.tick_params(axis='x', labelrotation=90)
ax.set_title(f"{mid_cardinality[1]} countplot", fontweight="bold")

plt.show()

In [ ]:
plt.figure(figsize=(20, 8))

# Plotting the countplot of mid_cardinality columns
ax = sns.countplot(data=train, x=mid_cardinality[2], hue="is_purchase")
ax.set_ylabel("Count", fontweight="bold")
ax.set_xlabel(mid_cardinality[2], fontweight="bold")
ax.tick_params(axis='x', labelrotation=90)
ax.set_title(f"{mid_cardinality[2]} countplot", fontweight="bold")

plt.show()

# Chi-Square Test

In [ ]:
from scipy.stats import chi2_contingency

# Creating a function called "cramer" to find the correlation of categorical columns with target
def cramer(col, target):
    contingency_table = pd.crosstab(col, target)
    (chi2, _, _, _) = chi2_contingency(contingency_table)
    total_sum = contingency_table.sum().sum()
    (x, y) = contingency_table.shape
    ans = np.sqrt(chi2/(total_sum*(min(x, y)-1)))
    return ans

### Low Cardinality Features Correlation

In [ ]:
# Getting the correlation of "low_cardinality" columns using the "cramer" function
corr = [cramer(train[col], train["purchaseValue"]) for col in low_cardinality]

# Print the columns with correlation
for i in range(len(corr)):
    print(f"{low_cardinality[i]} : {corr[i]}")

# Plot the columns and correlation with target
sns.barplot(x=low_cardinality, y=corr, width=0.5)
plt.title("Correlation of Low Cardinality Columns with Target", fontweight="bold")
plt.xlabel("Features", fontweight="bold")
plt.ylabel("Correlation", fontweight="bold")
plt.xticks(rotation=30, ha="right")

plt.show()

### Medium Cardinality Features Correlation

In [ ]:
# Getting the correlation of "mid_cardinality" columns using the "cramer" function
corr = [cramer(train[col], train["purchaseValue"]) for col in mid_cardinality]

# Print the columns with correlation
for i in range(len(corr)):
    print(f"{mid_cardinality[i]} : {corr[i]}")

# Plot the columns and correlation with target
sns.barplot(x=mid_cardinality, y=corr, width=0.5)
plt.title("Correlation of Mid Cardinality Columns with Target", fontweight="bold")
plt.xlabel("Features", fontweight="bold")
plt.ylabel("Correlation", fontweight="bold")
plt.xticks(rotation=30, ha="right")

plt.show()

### High Cardinality Features Correlation

In [ ]:
# Getting the correlation of "high_cardinality" columns using the "cramer" function
corr = [cramer(train[col], train["purchaseValue"]) for col in high_cardinality]

# Print the columns with correlation
for i in range(len(corr)):
    print(f"{high_cardinality[i]} : {corr[i]}")

# Plot the columns and correlation with target
sns.barplot(x=high_cardinality, y=corr, width=0.5)
plt.title("Correlation of High Cardinality Columns with Target", fontweight="bold")
plt.xlabel("Features", fontweight="bold")
plt.ylabel("Correlation", fontweight="bold")
plt.xticks(rotation=30, ha="right")

plt.show()

---

# Feature Engineering ⚙️

In [ ]:
# Droppint the following columns
cols_to_drop = ["sessionId", "sessionStart", "userId", "weekofyear", "day"]

train = train.drop(columns=cols_to_drop, axis=1)
test = test.drop(columns=cols_to_drop, axis=1)

In [ ]:
# get os occuring less than 100 times
os_counts = train['os'].value_counts()
less_os = os_counts[os_counts < 100].index

# replace all occurences of "less_os" with "others"
train['os'] = train['os'].replace(less_os, "others")

In [ ]:
# replacing "(none)" with "direct"
train["trafficSource.medium"] = train["trafficSource.medium"].replace("(none)", "direct")
test["trafficSource.medium"] = test["trafficSource.medium"].replace("(none)", "direct")

In [ ]:
# replacing "affiliate", "cpc" and "cpm" with "other"
train["trafficSource.medium"] = train["trafficSource.medium"].replace(["affiliate", "cpc", "cpm"], "other")
test["trafficSource.medium"] = test["trafficSource.medium"].replace(["affiliate", "cpc", "cpm"], "other")
train["trafficSource.medium"].value_counts()

In [ ]:
# replacing "Africa" and "Oceania" with "other"
train["geoNetwork.continent"] = train["geoNetwork.continent"].replace(["Africa", "Oceania"], "other")
test["geoNetwork.continent"] = test["geoNetwork.continent"].replace(["Africa", "Oceania"], "other")
test["geoNetwork.continent"].value_counts()

In [ ]:
# replacing "Paid Search", "Display", "Affiliates" and "(Other)" with "other"
train["userChannel"] = train["userChannel"].replace(["Paid Search", "Display", "Affiliates", "(Other)"], "other")
test["userChannel"] = test["userChannel"].replace(["Paid Search", "Display", "Affiliates", "(Other)"], "other")
train["userChannel"].value_counts()

In [ ]:
# Adding new feature with "userChannel" & "trafficSource.medium"
train["channel_medium"] = train["userChannel"] + "_" + train["trafficSource.medium"]
test["channel_medium"] = test["userChannel"] + "_" + test["trafficSource.medium"]
train["channel_medium"].value_counts()

In [ ]:
# Adding new feature with "userChannel" & "deviceType"
train["channel_device"] = train["userChannel"] + "_" + train["deviceType"]
test["channel_device"] = test["userChannel"] + "_" + test["deviceType"]
train["channel_device"].value_counts()

In [ ]:
# Adding new feature with "userChannel" & "os"
train["channel_os"] = train["userChannel"] + "_" + train["os"]
test["channel_os"] = test["userChannel"] + "_" + test["os"]
train["channel_os"].value_counts()

In [ ]:
# Adding new feature with "userChannel" & "device.isMobile" and typecasting Bool to String
train["channel_isMobile"] = train["userChannel"] + "_" + train["device.isMobile"].astype(str)
test["channel_isMobile"] = test["userChannel"] + "_" + test["device.isMobile"].astype(str)
train["channel_isMobile"].value_counts()

In [ ]:
# Adding new feature with "userChannel" & "is_weekend" and typecasting Num to String
train["channel_isWeekend"] = train["userChannel"] + "_" + train["is_weekend"].astype(str)
test["channel_isWeekend"] = test["userChannel"] + "_" + test["is_weekend"].astype(str)
train["channel_isWeekend"].value_counts()

In [ ]:
# Adding new feature with "userChannel" & "totals.bounces" and typecasting Num to String
train["channel_bounces"] = train["userChannel"] + "_" + train["totals.bounces"].astype(str)
test["channel_bounces"] = test["userChannel"] + "_" + test["totals.bounces"].astype(str)
train["channel_bounces"].value_counts()

---

# Target Variable

### 1. Target variable `purchaseValue` is zero inflated and highly skewed with `79.32%` values being `0`
### 2. Instances of `purchaseValue` where a purchase was made, skewness is `26.37` and kurtosis is `1067.51`
### 3. Log transformation significantly reduces the skewness to `0.37` and kurtosis to `1.10`
### 4. Yeo-Johnson transformation further reduces the skewness to `-0.03` and kurtosis to `1.67`

### ✅ CONCLUSION

### `Although, Log Transformation and Yeo-Johnson drastically reduce the skewness and kurtosis, but they compress the target variable's scale. This can also amplify the errors when reverting from log/yeo scale to the original scale`

### `These tranformations works better when the target variable doesn't span a large numeric range. In a case like this, a model like XGBoost or LightGBM with tweedie objective might perform well on the non-transformed target`

In [ ]:
# Printing skewness and kurtosis of target variable
print("Skewness:", round(train["purchaseValue"].skew(), 2))
print("Kurtosis:", round(train["purchaseValue"].kurt(), 2))
print(train["purchaseValue"].describe())

In [ ]:
# Printing the number of zero instances and their percentage
print("Number of zero values:", (train["purchaseValue"] == 0).sum())
print("Percentage of zero values:", (train["purchaseValue"] == 0).mean() * 100)

In [ ]:
# Skewness and Kurtosis of target variable where a purchase was made
print("Target variable purchased instances skewness :", round(train[train["purchaseValue"]>0]["purchaseValue"].skew(), 2))
print("Target variable purchased instances kurtosis :", round(train[train["purchaseValue"]>0]["purchaseValue"].kurt(), 2))

### Comparison with Transformations

In [ ]:
from scipy.stats import yeojohnson, skew, kurtosis, boxcox

# Retrieving the instances where a purchase was made
purchased = train[train["purchaseValue"]>0]["purchaseValue"]
log_purchased = np.log1p(purchased)
yeo_purchased = yeojohnson(purchased, -0.07)

In [ ]:
plt.figure(figsize=(12, 5))

# Plotting both log and yeo transformations
sns.histplot(x=log_purchased, bins=50, color="blue", label="Log", kde=True)
sns.histplot(x=yeo_purchased, bins=50, color="green", label="Yeo Johnson", kde=True)
plt.legend()
plt.tight_layout()
plt.title("Target Variable Transformations", fontweight="bold")
plt.show()

In [ ]:
# skew
purchased_skew = round(purchased.skew(), 2)
log_purchased_skew = round(log_purchased.skew(), 2)
yeo_purchased_skew = round(skew(yeojohnson(purchased, -0.07)), 2)

# kurtosis
purchased_kurt = round(purchased.kurt(), 2)
log_purchased_kurt = round(log_purchased.kurt(), 2)
yeo_purchased_kurt = round(kurtosis(yeojohnson(purchased, -0.07)), 2)

# skew_df = pd.DataFrame(data=[[purchased_skew, log_purchased_skew, yeo_purchased_skew], [purchased_skew, log_purchased_skew, yeo_purchased_skew]], columns=["Skew", "Kurtosis"], index=["Original", "Log", "Yeo Johnson"])
dist_df = pd.DataFrame(index=["Original", "Log", "Yeo Johnson"])
dist_df["Skew"] = [purchased_skew, log_purchased_skew, yeo_purchased_skew]
dist_df["Kurtosis"] = [purchased_kurt, log_purchased_kurt, yeo_purchased_kurt]

dist_df

---

# Two-Staged Model

### 1. Target variable is **Zero Inflated** and **Heavily Skewed**
### 2. Approx. `80%` of the values in target variable are `0` and the other `20%` has very high ranging values
### 3. Apply a classification model to differentiate between `purchase` and `non-purchase` instances.
### 4. Then, train the regression model on the `purchase` instances.
### 5. Test the regression model on train and validation set using metrics like `r2_score`, `mae`, `rmse`
### 6. Perform hyperparameter tuning on the model to help model generalize well.
### 5. At the end, predict the instances of `purchase` on test data and then predict the `purchaseValue` using the trained regression model.

## Stage-1 : Classification 0️⃣1️⃣

### 1. Training the classification model on `is_purchase`
### 2. `mean` impuation on numeric columns and `most_frequent` imputation on categorical columns
### 3. `RobustScaler` on numeric columns and `OrdinalEncoder` on categorical columns

In [ ]:
# Keeping "is_purchase" feature to train the classification model
X_class = train.drop(columns=["purchaseValue", "is_purchase"], axis=1)
y_class = train["is_purchase"]

In [ ]:
# Splitting the data into test and validation set
from sklearn.model_selection import train_test_split
X_train_cls, X_val_cls, y_train_cls, y_val_cls = train_test_split(X_class, y_class, test_size=0.2, stratify=y_class, random_state=42)
X_train_cls.shape, X_val_cls.shape, y_train_cls.shape, y_val_cls.shape

In [ ]:
# Retrieving the numerical and categorical columns
num_cols = X_train_cls.select_dtypes(exclude="object").columns
cat_cols = X_train_cls.select_dtypes(include="object").columns

len(num_cols), len(cat_cols)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Mean imputation and RobustScaler for numerical columns
num_pipe = Pipeline([
    ("num_imputer", SimpleImputer(strategy="mean")),
    ("num_scaler", RobustScaler())
])

# Mode imputation and OrdinalEncoder for categorical columns
cat_pipe = Pipeline([
    ("cat_imputer", SimpleImputer(strategy="most_frequent")),
    ("cat_encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

# Combining the above two processes
col_pipe = ColumnTransformer([
    ("num_process", num_pipe, num_cols),
    ("cat_process", cat_pipe, cat_cols)
])

col_pipe

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Classifier model
model = XGBClassifier()

# Column transformation and model pipeline
clf = Pipeline([
    ("transformation", col_pipe),
    ("model", model)
])

# Model training
clf.fit(X_train_cls, y_train_cls)

# Predicition
y_val_pred = clf.predict(X_val_cls)

# Scoring
print(classification_report(y_val_cls, y_val_pred))
print()
print(confusion_matrix(y_val_cls, y_val_pred))

---

## Stage-2 : Regression 📈

### 1. Training the regression model instances where a purchase was made with `purchaseValue`
### 2. `mean` impuation on numeric columns and `most_frequent` imputation on categorical columns
### 3. `RobustScaler` on numeric columns and `OrdinalEncoder` on categorical columns
### 4. Second degree polynomial transformation on numeric columns for feature interactions

In [ ]:
# preparing to train regression model only on instances where a purchase was made
train_reg = train[train["purchaseValue"]>0].copy()

# Keeping "purchaseValue" feature to train the regression model
X_reg = train_reg.drop(columns=["purchaseValue", "is_purchase"], axis=1)
y_reg = train_reg["purchaseValue"]

In [ ]:
# Splitting the data into train and validation set
from sklearn.model_selection import train_test_split

X_train_reg, X_val_reg, y_train_reg, y_val_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)
X_train_reg.shape, X_val_reg.shape, y_train_reg.shape, y_val_reg.shape

In [ ]:
# Retrieving the numerical and categorical columns
num_cols_reg = X_train_reg.select_dtypes(exclude="object").columns
cat_cols_reg = X_train_reg.select_dtypes(include="object").columns

len(num_cols_reg), len(cat_cols_reg)

In [ ]:
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler

# creating degree=2 poly. features for the numeric columns
num_pipe_reg = Pipeline([
    ("num_imputer", SimpleImputer(strategy="mean")),
    ("num_poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("num_scaler", RobustScaler())
])

# Mode imputation and OrdinalEncoder for categorical columns
cat_pipe_reg = Pipeline([
    ("cat_imputer", SimpleImputer(strategy="most_frequent")),
    ("cat_encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
])

# Combining the above two processes
col_pipe_reg = ColumnTransformer([
    ("num_process", num_pipe_reg, num_cols_reg),
    ("cat_process", cat_pipe_reg, cat_cols_reg)
])

col_pipe_reg

# Model-1 : XGBoost

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, median_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV

# # model
# model = XGBRegressor(
#     objective="reg:tweedie",
#     tweedie_variance_power=1.1,
#     n_jobs=-1,
#     random_state=42,
#     tree_method="hist",
# )

# # preprocessor and model
# regressor = Pipeline([
#     ("preprocessor", col_pipe_reg),
#     ("model", model)
# ])

# # parameter space for XGBoost hypertuning
# params = {
#     "model__n_estimators": [400, 450, 500, 550, 600, 650, 700, 750, 800, 850],
#     "model__max_depth": [5, 6, 7, 8],
#     "model__learning_rate": [0.01, 0.02, 0.03, 0.04, 0.05, 0.06],
#     "model__subsample": [0.5, 0.7, 1],
#     "model__colsample_bytree": [0.01, 0.02, 0.03, 0.04, 0.05, 0.06],
#     "model__reg_lambda": [2, 3, 4, 5, 6]
# }

# search = RandomizedSearchCV(
#     regressor,
#     params,
#     cv=5,
#     n_iter=20,
#     random_state=42,
#     verbose=1,
#     scoring="neg_mean_absolute_error",
#     n_jobs=-1
# )

# Best parameters for hyper tuned XGBoost Model
best_params = {
    'subsample': 1,
    'reg_lambda': 3,
    'n_estimators': 800,
    'max_depth': 8,
    'learning_rate': 0.06,
    'colsample_bytree': 0.06
}

# model
model = XGBRegressor(
    objective="reg:tweedie",
    tweedie_variance_power=1.1,
    n_jobs=-1,
    random_state=42,
    tree_method="hist",
    **best_params
)

# Column transformation and best model pipeline
best_model = Pipeline([
    ("preprocessor", col_pipe_reg),
    ("model", model)
])

# Model Training
best_model.fit(X_train_reg, y_train_reg)

# Prediction
y_pred_train = best_model.predict(X_train_reg)
y_pred_val = best_model.predict(X_val_reg)

# Scoring
train_score = round(r2_score(y_train_reg, y_pred_train) * 100, 2)
val_score = round(r2_score(y_val_reg, y_pred_val) * 100, 2)
val_mae = round(mean_absolute_error(y_val_reg, y_pred_val), 2)
val_rmse = round(np.sqrt(mean_squared_error(y_val_reg, y_pred_val)), 2)
val_medae = round(median_absolute_error(y_val_reg, y_pred_val), 2)

# Printing the scoring metrics
print("Train R2 :", train_score)
print("Val R2:", val_score)
print()
print("Val MAE:", val_mae)
print("Val RMSE:", val_rmse)
print("Val Median AE:", val_medae)

# Model-2 : Random Forest Regressor

In [ ]:
# from sklearn.ensemble import RandomForestRegressor

# # # model
# # model = RandomForestRegressor(
# #     random_state=42,
# #     n_jobs=-1
# # )

# # # preprocessor and model
# # regressor = Pipeline([
# #     ("preprocessor", col_pipe_reg),
# #     ("model", model)
# # ])

# # # parameter space for XGBoost hypertuning
# # params = {
# #     'model__max_depth': [16, 17, 18],
# #     'model__n_estimators': [400, 600, 800],
# #     'model__min_samples_split': [2, 5, 10],
# #     'model__min_samples_leaf': [1, 2, 4],
# #     'model__max_features': ['sqrt', 0.05, 0.1, 0.2, 1.0],
# # }

# # search = RandomizedSearchCV(
# #     regressor,
# #     params,
# #     cv=5,
# #     n_iter=20,
# #     random_state=42,
# #     verbose=1,
# #     scoring="neg_mean_absolute_error",
# #     n_jobs=-1
# # )

# # # training the model
# # search.fit(X_train_reg, y_train_reg)

# # best_model = search.best_estimator_

# best_params = {
#     'n_estimators': 600,
#     'min_samples_split': 2,
#     'min_samples_leaf': 1,
#     'max_features': 'sqrt',
#     'max_depth': 17
# }

# # model
# model = RandomForestRegressor(
#     random_state=42,
#     n_jobs=-1,
#     **best_params
# )

# # Column transformation and best model pipeline
# best_model = Pipeline([
#     ("preprocessor", col_pipe_reg),
#     ("model", model)
# ])

# # Model Training
# best_model.fit(X_train_reg, y_train_reg)

# # Prediction
# y_pred_train = best_model.predict(X_train_reg)
# y_pred_val = best_model.predict(X_val_reg)

# # Scoring
# train_score = round(r2_score(y_train_reg, y_pred_train) * 100, 2)
# val_score = round(r2_score(y_val_reg, y_pred_val) * 100, 2)
# val_mae = round(mean_absolute_error(y_val_reg, y_pred_val), 2)
# val_rmse = round(np.sqrt(mean_squared_error(y_val_reg, y_pred_val)), 2)
# val_medae = round(median_absolute_error(y_val_reg, y_pred_val), 2)

# Printing the scoring metrics
# print("Train R2 :", train_score)
# print("Val R2:", val_score)
# print()
# print("Val MAE:", val_mae)
# print("Val RMSE:", val_rmse)
# print("Val Median AE:", val_medae)

# Model-3 : LightGBM

In [ ]:
# from lightgbm import LGBMRegressor

# # # model
# # model = LGBMRegressor(
# #     objective="tweedie",
# #     tweedie_variance_power=1.1,
# #     metric="mae",
# #     random_state=42,
# #     n_jobs=-1
# # )

# # # preprocessor and model
# # regressor = Pipeline([
# #     ("preprocessor", col_pipe_reg),
# #     ("model", model)
# # ])

# # # parameter space for XGBoost hypertuning
# # params = {
# #     "model__n_estimators": [400, 500, 600],
# #     "model__max_depth": [6, 7, 8, 9],
# #     "model__learning_rate": [0.01, 0.03, 0.05],
# #     "model__subsample": [0.1, 0.2, 0.3, 0.4, 0.5],
# #     "model__colsample_bytree": [0.6, 0.8, 1.0],
# #     "model__reg_lambda": [1, 2, 4]
# # }

# # search = RandomizedSearchCV(
# #     regressor,
# #     params,
# #     cv=2,
# #     random_state=42,
# #     verbose=1,
# #     scoring="neg_mean_absolute_error",
# #     n_jobs=-1
# # )

# # # training the model
# # search.fit(X_train_reg, y_train_reg)

# # best_model = search.best_estimator_

# best_params = {
#     'subsample': 0.2,
#     'reg_lambda': 1,
#     'n_estimators': 500,
#     'max_depth': 8,
#     'learning_rate': 0.05,
#     'colsample_bytree': 0.6
# }

# # model
# model = LGBMRegressor(
#     objective="tweedie",
#     tweedie_variance_power=1.1,
#     metric="mae",
#     random_state=42,
#     n_jobs=-1,
#     **best_params
# )

# # Column transformation and best model pipeline
# best_model = Pipeline([
#     ("preprocessor", col_pipe_reg),
#     ("model", model)
# ])

# # Training the model
# best_model.fit(X_train_reg, y_train_reg)

# # Prediction
# y_pred_train = best_model.predict(X_train_reg)
# y_pred_val = best_model.predict(X_val_reg)

# # Scoring
# train_score = round(r2_score(y_train_reg, y_pred_train) * 100, 2)
# val_score = round(r2_score(y_val_reg, y_pred_val) * 100, 2)
# val_mae = round(mean_absolute_error(y_val_reg, y_pred_val), 2)
# val_rmse = round(np.sqrt(mean_squared_error(y_val_reg, y_pred_val)), 2)
# val_medae = round(median_absolute_error(y_val_reg, y_pred_val), 2)

# Printing the scoring metrics
# print("Train R2 :", train_score)
# print("Val R2:", val_score)
# print()
# print("Val MAE:", val_mae)
# print("Val RMSE:", val_rmse)
# print("Val Median AE:", val_medae)

# Model-4 : Voting Regressor

In [ ]:
# from sklearn.ensemble import VotingRegressor

# # XGBoost best params
# xgb_best_params = {
#     'subsample': 1,
#     'reg_lambda': 3,
#     'n_estimators': 800,
#     'max_depth': 8,
#     'learning_rate': 0.06,
#     'colsample_bytree': 0.06
# }

# # XGBoost Model
# xgb_model = XGBRegressor(
#     objective="reg:tweedie",
#     tweedie_variance_power=1.1,
#     n_jobs=-1,
#     random_state=42,
#     tree_method="hist",
#     **xgb_best_params
# )

# # RandomForest best params
# rf_best_params = {
#     'n_estimators': 600,
#     'min_samples_split': 2,
#     'min_samples_leaf': 1,
#     'max_features': 'sqrt',
#     'max_depth': 17
# }

# # RandomForest Model
# rf_model = RandomForestRegressor(
#     random_state=42,
#     n_jobs=-1,
#     **rf_best_params
# )

# # Setting up voting regressor
# voting_model = VotingRegressor(estimators=[
#     ("xgb", xgb_model),
#     ("rf", rf_model)
# ], weights=[0.9, 0.1])

# # Column transformation and best model pipeline
# best_model = Pipeline([
#     ("preprocessor", col_pipe_reg),
#     ("model", voting_model)
# ])

# # # Finding optimal weights for xgb and lbg models
# # for i in np.arange(0.1, 1.0, 0.1):
# #     weights = [round(i, 2), round(1-i, 2)]
# #     voting_model.set_params(weights=weights)

# #     best_model.fit(X_train_reg, y_train_reg)
# #     y_pred_train = best_model.predict(X_train_reg)
# #     y_pred_val = best_model.predict(X_val_reg)

# #     train_score = round(r2_score(y_train_reg, y_pred_train) * 100, 2)
# #     val_score = round(r2_score(y_val_reg, y_pred_val) * 100, 2)
# #     val_mae = round(mean_absolute_error(y_val_reg, y_pred_val), 2)

# #     print(weights)
# #     print("Train R2 :", train_score)
# #     print("Val R2:", val_score)
# #     print("Val MAE:", val_mae)
# #     print()

# # Training the model
# best_model.fit(X_train_reg, y_train_reg)

# # Prediction
# y_pred_train = best_model.predict(X_train_reg)
# y_pred_val = best_model.predict(X_val_reg)

# # Scoring
# train_score = round(r2_score(y_train_reg, y_pred_train) * 100, 2)
# val_score = round(r2_score(y_val_reg, y_pred_val) * 100, 2)
# val_mae = round(mean_absolute_error(y_val_reg, y_pred_val), 2)
# val_rmse = round(np.sqrt(mean_squared_error(y_val_reg, y_pred_val)), 2)
# val_medae = round(median_absolute_error(y_val_reg, y_pred_val), 2)

# Printing the scoring metrics
# print("Train R2 :", train_score)
# print("Val R2:", val_score)
# print()
# print("Val MAE:", val_mae)
# print("Val RMSE:", val_rmse)
# print("Val Median AE:", val_medae)

# Model Performance 🎖️
## 🥇 XGBoost
## 🥈 VotingRegressor
## 🥉 RandomForestRegressor
## ⭐ LightGBM

In [ ]:
# Models
models = ["XGBoost", "RandomForestRegressor", "LightGBM", "VotingRegressor"]

# R2 scores
scores = [0.61542, 0.42986, 0.37655, 0.60874]

# Mean Absolute Error
mae = [76931718.21, 100769415.76, 93551627.94, 78388776.81]

# Root Mean Square Error
rmse = [201219302.19, 289230005.23, 233289187.38, 205213309.45]

# Median Absolute Error
med_ae = [34480392.0, 49556693.94, 44680192.82, 35803392.79]

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
axes = axes.flatten()

# plotting the metrics
axes[0].bar(models, scores, width=0.5, color='green')
axes[0].set_title("R2 Scores", fontweight="bold")

axes[1].bar(models, mae, width=0.5, color='red')
axes[1].set_title("MAE", fontweight="bold")

axes[2].bar(models, rmse, width=0.5, color='blue')
axes[2].set_title("RMSE", fontweight="bold")

axes[3].bar(models, med_ae, width=0.5, color='orange')
axes[3].set_title("Median AE", fontweight="bold")

plt.tight_layout()

plt.show()

In [ ]:
# checking if predictions for submission.csv are made with correct setup
best_model

# Final Submission

In [ ]:
try:
    X_test = test.copy()

    # Predict purchases
    y_test_cls_pred = clf.predict(X_test)

    # Get X_test rows where a purchase is predicted
    X_test_reg_raw = X_test[y_test_cls_pred == 1].copy()

    # Predict with regression on raw rows (preprocessing + model)
    y_test_reg_pred = best_model.predict(X_test_reg_raw)

    # Check for NaN or inf in regression output
    y_test_reg_pred = np.nan_to_num(y_test_reg_pred, nan=0.0, posinf=0.0, neginf=0.0)

    # Final predictions
    final_predictions = np.zeros(len(X_test))
    final_predictions[y_test_cls_pred == 1] = y_test_reg_pred

    # Creating submission dataframe
    submission = pd.DataFrame({
        "id": range(0, test.shape[0]),
        "target": final_predictions
    })

    # Creating submission.csv
    submission.to_csv("submission.csv", index=False)

    print("Success!")

except Exception as e:
    print("Error :", e)